In [37]:
import pandas as pd
import numpy as np

In [38]:
df = pd.read_pickle('data_clean.pkl')
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60


In [39]:
actual_date = df['InvoiceDate'].max() + pd.Timedelta(days = 1)
actual_date

Timestamp('2011-12-10 12:50:00')

In [40]:
df['Revenue'].info() 

<class 'pandas.Series'>
Index: 392692 entries, 0 to 541908
Series name: Revenue
Non-Null Count   Dtype  
--------------   -----  
392692 non-null  float64
dtypes: float64(1)
memory usage: 6.0 MB


In [41]:
RFM = df.groupby('CustomerID').agg(
    Recency = ('InvoiceDate', lambda x: (actual_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary = ('Revenue', 'sum')
)
RFM.head(369)

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40
...,...,...,...
12809.0,178,1,489.31
12811.0,261,3,514.85
12812.0,45,1,229.64


In [42]:
RFM['R_score'] = pd.qcut(RFM['Recency'], 5, labels = [5,4,3,2,1])
RFM['F_score'] = pd.qcut(RFM['Frequency'].rank(method ='first'), 5, labels = [1,2,3,4,5])
RFM['M_score'] = pd.qcut(RFM['Monetary'], 5, labels = [1,2,3,4,5])
RFM['RFM_Score_sum'] = RFM['R_score'].astype(int) + RFM['F_score'].astype(int) + RFM['M_score'].astype(int)
RFM['RFM_Score_str'] = RFM['R_score'].astype(str) + RFM['F_score'].astype(str) + RFM['M_score'].astype(str)
RFM

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score_sum,RFM_Score_str
CustomerID,,,,,,,,
12346.0,326,1,77183.60,1,1,5,7,115
12347.0,2,7,4310.00,5,5,5,15,555
12348.0,75,4,1797.24,2,4,4,10,244
12349.0,19,1,1757.55,4,1,4,9,414
12350.0,310,1,334.40,1,1,2,4,112
...,...,...,...,...,...,...,...,...
18280.0,278,1,180.60,1,2,1,4,121
18281.0,181,1,80.82,1,2,1,4,121
18282.0,8,2,178.05,5,3,1,9,531


In [43]:
def identification(row):
    if (row['R_score'] == 5) and (row['F_score'] == 5) and (row['M_score'] == 5):
        return 'Champion'
    elif (row['F_score'] in [4,5]) and (row['M_score'] in [4,5]) and (row['R_score'] in [1,2]):
        return 'At Risk'
    elif (row['F_score'] in [1,2]) and (row['M_score'] in [1,2]) and (row['R_score'] in [1,2]):
        return 'Lost'
    elif (row['R_score'] in [4,5]) and (row['F_score'] in [1,2]):
        return 'New'
    elif (row['R_score'] in [4,5]) and (row['M_score'] in [2,3]) and (row['F_score'] in [2,3]):
        return 'Potential'
    elif (row['F_score'] in [4,5]) and (row['M_score'] in [2,3,4]) and (row['R_score'] in [2,3,4]):
        return 'Loyal'
    elif (row['F_score'] in [4,5]) and (row['M_score'] in [4,5]) and (row['R_score'] == 1):
        return 'Cant Lose'
    else:
        return 'Other'
RFM['Segment'] = RFM.apply(identification, axis = 1)
RFM

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score_sum,RFM_Score_str,Segment
CustomerID,,,,,,,,,
12346.0,326,1,77183.60,1,1,5,7,115,Other
12347.0,2,7,4310.00,5,5,5,15,555,Champion
12348.0,75,4,1797.24,2,4,4,10,244,At Risk
12349.0,19,1,1757.55,4,1,4,9,414,New
12350.0,310,1,334.40,1,1,2,4,112,Lost
...,...,...,...,...,...,...,...,...,...
18280.0,278,1,180.60,1,2,1,4,121,Lost
18281.0,181,1,80.82,1,2,1,4,121,Lost
18282.0,8,2,178.05,5,3,1,9,531,Other


In [44]:
RFM['Segment'].value_counts('Champion')

Segment
Other        0.448824
Lost         0.189258
Loyal        0.126095
Champion     0.079991
New          0.073536
Potential    0.043568
At Risk      0.038728
Name: proportion, dtype: float64

In [45]:
RFM.to_pickle('RFM анализ таблица.pkl')

In [46]:
RFM = RFM.groupby('Segment').agg(
    Client_count =('Recency', 'count'),
    Revenue_all = ('Monetary', 'sum'),
    Avg_check = ('Monetary','mean')
)
RFM.head()

,Client_count,Revenue_all,Avg_check
Segment,,,
At Risk,168,370498.850,2205.350298
Champion,347,3896691.140,11229.657464
Lost,821,187629.491,228.537748
Loyal,547,597167.791,1091.714426
New,319,145219.740,455.234295


In [47]:
RFM['Revenue_piece_%'] = (RFM['Revenue_all'] / RFM['Revenue_all'].sum())* 100
RFM

,Client_count,Revenue_all,Avg_check,Revenue_piece_%
Segment,,,,
At Risk,168,370498.850,2205.350298,4.168900
Champion,347,3896691.140,11229.657464,43.846062
Lost,821,187629.491,228.537748,2.111231
Loyal,547,597167.791,1091.714426,6.719408
New,319,145219.740,455.234295,1.634031
Other,1947,3588252.382,1842.964757,40.375470
Potential,189,101749.500,538.357143,1.144898


In [49]:
# 1. Самые прибыльные базы это 'Champion' и 'Other'.
# 2. 8% чемпионов приносят почти 44 процента выручки компании
# 3. Существует серьезная проблема с удержанием клиентов, почти четверть клиентов потеряны либо же находятся в спячке и приносят всего 2% выручки